In [1]:
import os

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"
os.environ["PATH"] += ";" + r"C:\hadoop\bin"

In [6]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

import pandas as pd
import duckdb
from datetime import datetime

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

import time

start_time = time.time()

builder = (
    SparkSession.builder
    .master("local[*]")
    .appName("MegaMart")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
    .config(
        "spark.hadoop.hadoop.home.dir",
        r"C:\hadoop"
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()


spark.sparkContext._jsc.hadoopConfiguration().set(
    "hadoop.home.dir",
    r"C:\hadoop"
)

run_id = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


In [13]:
BASE_PATH = "../megamart/megamart_dataset"

customer_snapshot = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{BASE_PATH}/customers.csv")
)

transactions = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{BASE_PATH}/transactions.csv")
)

In [8]:
for tbl in [
    customer_snapshot
]:
    tbl.printSchema()
    tbl.show(5, False)

root
 |-- customer_id: string (nullable = true)
 |-- full_name: string (nullable = true)
 |-- nik: long (nullable = true)
 |-- phone: long (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- province: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- registration_channel: string (nullable = true)
 |-- loyalty_tier: string (nullable = true)
 |-- lifetime_value: double (nullable = true)
 |-- preferred_store_id: string (nullable = true)
 |-- consent_marketing: boolean (nullable = true)
 |-- consent_analytics: boolean (nullable = true)
 |-- consent_third_party: boolean (nullable = true)
 |-- churn_risk_score: double (nullable = true)
 |-- first_purchase_date: date (nullable = true)
 |-- last_purchase_date: date (nullable = true)

+-----------+----------+----------------+-------------+-------------------+--------+----------------+-----------------+--------------------+------------+--------------+------------------+------

In [36]:
# =====================================================
# SCD2 CONFIG
# =====================================================

TODAY = F.current_date()
MAX_DATE = F.lit("9999-12-31").cast("date")

# =====================================================
# TRY LOAD EXISTING DIMENSION
# =====================================================

try:

    dim_customer = (
        spark.read
        .format("jdbc")
        .option("url", "jdbc:duckdb:megamart.duckdb")
        .option("dbtable", "gold.dim_customers_scd2")
        .load()
    )

    first_load = False

except:

    first_load = True

# =====================================================
# INITIAL LOAD
# =====================================================

if first_load:

    dim_customer = (

        customer_snapshot

        .withColumn(
            "surrogate_key",
            F.monotonically_increasing_id() + 1
        )

        .withColumn(
            "effective_date",
            F.current_date()
        )

        .withColumn(
            "expiry_date",
            MAX_DATE
        )

        .withColumn(
            "is_current",
            F.lit(1)
        )

        .select(
            "surrogate_key",
            "customer_id",
            "full_name",
            "city",
            "loyalty_tier",
            "effective_date",
            "expiry_date",
            "is_current"
        )
    )

# =====================================================
# INCREMENTAL SCD2 MERGE
# =====================================================

else:

    current_dim = (
        dim_customer
        .filter(F.col("is_current") == 1)
    )

    compare_df = (

        customer_snapshot.alias("src")

        .join(
            current_dim.alias("tgt"),
            "customer_id",
            "left"
        )
    )

    # =========================================
    # NEW CUSTOMER
    # =========================================

    new_customer = (

        compare_df

        .filter(
            F.col("tgt.customer_id").isNull()
        )

        .select("src.*")
    )

    # =========================================
    # CHANGED CUSTOMER
    # =========================================

    changed_customer = (

        compare_df

        .filter(
            (F.col("src.city")
             != F.col("tgt.city"))

            |

            (F.col("src.loyalty_tier")
             != F.col("tgt.loyalty_tier"))
        )
    )

    # =========================================
    # EXPIRE OLD RECORD
    # =========================================

    expired_records = (

        dim_customer.alias("hist")

        .join(
            changed_customer.select(
                "customer_id"
            ),
            "customer_id"
        )

        .filter(
            F.col("hist.is_current") == 1
        )

        .withColumn(
            "expiry_date",
            F.date_sub(
                F.col("registration_date"),
                1
            )
        )

        .withColumn(
            "is_current",
            F.lit(0)
        )
    )

    # =========================================
    # NEW VERSION
    # =========================================

    changed_new_version = (

        changed_customer

        .select("src.*")

        .withColumn(
            "surrogate_key",
            F.monotonically_increasing_id()
            + 1000000
        )

        .withColumn(
            "effective_date",
            F.col("registration_date")
        )

        .withColumn(
            "expiry_date",
            MAX_DATE
        )

        .withColumn(
            "is_current",
            F.lit(1)
        )

        .select(
            "surrogate_key",
            "customer_id",
            "full_name",
            "city",
            "loyalty_tier",
            "effective_date",
            "expiry_date",
            "is_current"
        )
    )

    # =========================================
    # INSERT NEW CUSTOMER
    # =========================================

    new_customer_insert = (

        new_customer

        .withColumn(
            "surrogate_key",
            F.monotonically_increasing_id()
            + 2000000
        )

        .withColumn(
            "effective_date",
            F.col("registration_date")
        )

        .withColumn(
            "expiry_date",
            MAX_DATE
        )

        .withColumn(
            "is_current",
            F.lit(1)
        )

        .select(
            "surrogate_key",
            "customer_id",
            "full_name",
            "city",
            "loyalty_tier",
            "effective_date",
            "expiry_date",
            "is_current"
        )
    )

    unchanged = (

        dim_customer.alias("d")

        .join(
            changed_customer.select(
                "customer_id"
            ),
            "customer_id",
            "left_anti"
        )
    )

    dim_customer = (

        unchanged

        .unionByName(expired_records)

        .unionByName(changed_new_version)

        .unionByName(new_customer_insert)
    )

# =====================================================
# SAVE TO DUCKDB
# =====================================================

customer_pd = dim_customer.toPandas()

conn = duckdb.connect("megamart.duckdb")

conn.execute("""
CREATE SCHEMA IF NOT EXISTS gold
""")

conn.register(
    "customer_scd2_df",
    customer_pd
)

conn.execute("""
CREATE OR REPLACE TABLE gold.dim_customers_scd2 AS
SELECT *
FROM customer_scd2_df
""")

print("SCD Type 2 table saved")

# =====================================================
# VALIDATION
# =====================================================

conn.sql("""
SELECT
    customer_id,
    city,
    loyalty_tier,
    effective_date,
    expiry_date,
    is_current
FROM gold.dim_customers_scd2
LIMIT 10
""").show()

conn.close()

SCD Type 2 table saved
┌─────────────┬───────────────┬──────────────┬────────────────┬─────────────┬────────────┐
│ customer_id │     city      │ loyalty_tier │ effective_date │ expiry_date │ is_current │
│   varchar   │    varchar    │   varchar    │      date      │    date     │   int32    │
├─────────────┼───────────────┼──────────────┼────────────────┼─────────────┼────────────┤
│ CUST0000001 │ Semarang      │ DIAMOND      │ 2026-06-22     │ 9999-12-31  │          1 │
│ CUST0000002 │ Surabaya      │ SILVER       │ 2026-06-22     │ 9999-12-31  │          1 │
│ CUST0000003 │ Makassar      │ SILVER       │ 2026-06-22     │ 9999-12-31  │          1 │
│ CUST0000004 │ Solo          │ SILVER       │ 2026-06-22     │ 9999-12-31  │          1 │
│ CUST0000005 │ Surabaya      │ BRONZE       │ 2026-06-22     │ 9999-12-31  │          1 │
│ CUST0000006 │ Jakarta Timur │ PLATINUM     │ 2026-06-22     │ 9999-12-31  │          1 │
│ CUST0000007 │ Binjai        │ SILVER       │ 2026-06-22     │ 999

In [20]:
import duckdb

conn = duckdb.connect("megamart.duckdb")

conn.execute("""
CREATE SCHEMA IF NOT EXISTS silver
""")

files = [
    "transactions"
]

for table in files:

    conn.execute(f"""
    CREATE OR REPLACE TABLE silver.{table} AS
    SELECT *
    FROM read_csv_auto(
        '../megamart/megamart_dataset/{table}.csv',
        HEADER=TRUE
    )
    """)

    print(f"Loaded silver.{table}")

print("Done")

Loaded silver.transactions
Done


In [23]:
conn.sql("""
SELECT COUNT(*)
FROM silver.transactions
""").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       137045 │
└──────────────┘



In [27]:
conn.sql("""
SELECT
    t.txn_id,
    t.customer_id,
    t.txn_date,
    d.loyalty_tier,
    d.city,
    d.effective_date,
    d.expiry_date
FROM silver.transactions t
JOIN gold.dim_customers_scd2 d
    ON t.customer_id = d.customer_id
   AND t.txn_date BETWEEN d.effective_date
                      AND d.expiry_date
LIMIT 20
""").show()

┌─────────┬─────────────┬──────────┬──────────────┬─────────┬────────────────┬─────────────┐
│ txn_id  │ customer_id │ txn_date │ loyalty_tier │  city   │ effective_date │ expiry_date │
│ varchar │   varchar   │   date   │   varchar    │ varchar │      date      │    date     │
└─────────┴─────────────┴──────────┴──────────────┴─────────┴────────────────┴─────────────┘
                                           0 rows                                         



In [30]:
conn.sql("""
SELECT
    MIN(txn_date),
    MAX(txn_date)
FROM silver.transactions
""").show()

┌───────────────┬───────────────┐
│ min(txn_date) │ max(txn_date) │
│     date      │     date      │
├───────────────┼───────────────┤
│ 2025-08-01    │ 2026-01-30    │
└───────────────┴───────────────┘



In [32]:
conn.sql("""
SELECT COUNT(*)
FROM silver.transactions t
JOIN gold.dim_customers_scd2 d
ON t.customer_id = d.customer_id
""").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        95939 │
└──────────────┘



In [33]:
conn.sql("""
SELECT
    MIN(effective_date),
    MAX(effective_date)
FROM gold.dim_customers_scd2
""").show()

┌─────────────────────┬─────────────────────┐
│ min(effective_date) │ max(effective_date) │
│        date         │        date         │
├─────────────────────┼─────────────────────┤
│ 2026-06-22          │ 2026-06-22          │
└─────────────────────┴─────────────────────┘

